# Interpolation and Extrapolation

```{note}
This lecture follows closely
[Numerical Recipes](https://numerical.recipes/) 2nd Edition in C and
3rd Edition in C++, Chapter 3 "Interpolation and Extrapolation".
```

In scientific computing and machine learning, interpolation and
extrapolation are essential methods for estimating unknown values from
known data.

Interpolation deals with predicting values within the range of
available data.
This is the foundation of most supervised learning tasks in machine
learning, where models predict outputs in the same region where they
were trained.
Standard interpolation methods include:
* Polynomial interpolation is flexible but can suffer from
  oscillations at the edges of the interval (Runge's phenomenon).
* Rational interpolation can help stabilize the behavior, especially
  near asymptotes or when functions have strong curvature.
* Spline interpolation, particularly cubic splines, provides smooth
  fits with continuity up to the second derivative.
  This is especially useful when a smooth curve is required, such as
  in modeling or visualization.

Extrapolation extends predictions beyond the available data.
It is inherently unreliable: without additional information, we cannot
know how a function continues beyond the known region.
A promising modern approach is physics-informed machine learning
(PIML), where physical laws such as ODEs are built into the model.
By respecting known constraints, such methods can make extrapolations
that remain consistent with physics.

Interpolation and function approximation are related but slightly
different.
It is useful to distinguish between them:
* Interpolation uses existing data to estimate specific missing
  values.
* Function approximation constructs a simplified function that
  captures the overall behavior of a more complex one.
  It is often for efficiency or analytic convenience.
  (See [Numerical Recipes](https://numerical.recipes/), Chapter 5.)

## Limitations of Interpolation

Even the best interpolation schemes can fail when the function itself
is ill-behaved.
For example:
\begin{align}
  f_\mathrm{sing}(x) = 3x^2 + \frac{1}{\pi^4}\ln\left[(\pi - x)^2\right] + 1
\end{align}
This function looks smooth but has a subtle singularity at $x = \pi$.
Interpolating only near the singularity produces misleading results,
as we see in these python plots.

In [ ]:
import numpy as np

def f_sing(x):
    return 3 * x**2 + np.log((np.pi - x)**2) / np.pi**4 + 1

x1 = np.linspace(3.13, 3.16, 3+1)     # very sparse
x2 = np.linspace(3.13, 3.16, 30+1)    # coarse
x3 = np.linspace(3.13, 3.16, 300+1)   # medium
x4 = np.linspace(3.13, 3.16, 3000+1)  # dense

In [ ]:
from matplotlib import pyplot as plt

plt.plot(x4, f_sing(x4),       label='3001 points')
plt.plot(x3, f_sing(x3), '--', label='301 points')
plt.plot(x2, f_sing(x2), 'o:', label='31 points')
plt.plot(x1, f_sing(x1), 'o-', label='4 points')
plt.legend()

This example shows why interpolation methods should always be paired
with error estimates and awareness of the underlying physics or
mathematics.

## Preliminaries: Searching an Ordered Table

Before we can interpolate, we need to know where in the dataset our
target value lies.
This step is called searching.
* If data are sampled on a regular grid, finding neighbors is trivial:
  just use the array index.
* If data are irregularly spaced, we must locate the two points that
  bracket the target value.
* This search step can be just as costly as the interpolation itself,
  so efficient methods are critical in practice.
  In fact, in multi-dimension, it may require non-trivial algorithm
  and advanced data structure.

[Numerical Recipes](https://numerical.recipes/) describes two main
approaches: bisection and hunting, each suited to different scenarios.


### Sequential Search

As a baseline, let's consider a simple sequential search.
It scans through the array until the first value larger than the
target is found.

In [ ]:
def sequential(X, v):
    for i in range(len(X)):  # use a Python loop for clarity
        if X[i] >= v:
            return i - 1

In [ ]:
import numpy as np

for _ in range(5):
    X = np.sort(np.random.uniform(0, 100, 10))
    v = np.random.uniform(min(X), max(X))
    i = sequential(X, v)
    assert X[i] <= v and v < X[i+1]
    print(f'{X[i]} <= {v} < {X[i+1]}')

This works, but it requires $\mathcal{O}(N)$ steps.
For large datasets, this is inefficient.

### Bisection Search

The bisection method is much faster.
It repeatedly halves the search interval until the target is bracketed.
For $N$ data points, it requires only about $\log_2(N)$ steps.

In [ ]:
def bisection(X, v):
    l, h = 0, len(X) - 1
    while h - l > 1:
        ...
    return l  # index of the closest value less than the target

In [ ]:
for _ in range(5):
    X = np.sort(np.random.uniform(0, 100, 10))
    v = np.random.uniform(min(X), max(X))
    i = bisection(X, v)
    assert X[i] <= v and v < X[i+1]
    print(f'{X[i]} <= {v} < {X[i+1]}')

This method is robust and efficient for uncorrelated queries, where
each target value is unrelated to the previous one.

### Hunting Method

If target values are requested in sequence and tend to be close to one
another, we can do even better.
The hunting method exploits this correlation:
1. Start near the last found index.
2. Step outward (doubling the step size each time) until the target is
   bracketed.
3. Refine the result using bisection in the narrowed interval.
This approach is often faster than starting from scratch with
bisection every time.

In [ ]:
def hunt(X, v, i_last):
    n = len(X)
    assert 0 <= i_last < n - 1

    if v >= X[i_last]:
        l, h, step = i_last, min(n-1, i_last+1), 1
        while ...:
            ...
            step *= 2
    else:
        l, h, step = max(0, i_last-1), i_last, 1
        while ...:
            ...
            step *= 2

    return bisection(X[l:h+1], v) + l

In [ ]:
X = np.sort(np.random.uniform(0, 100, 20))
V = np.linspace(min(X), max(X), 8, endpoint=False) + np.random.uniform(0, 1, 8)

i = len(X) // 2  # initial guess
for v in V:
    i = hunt(X, v, i)  # start from the last found index
    assert X[i] <= v and v < X[i+1]
    assert i == bisection(X, v)
    print(f'{i:2d}: {X[i]} <= {v} < {X[i+1]}')

## Linear Interpolation with Searching

With a search routine in place, interpolation becomes straightforward.
Below is a simple interpolator that supports hunt, bisection, or
sequential searching.

In [ ]:
class Interpolator:
    
    def __init__(self, X, Y):
        assert len(X) == len(Y)
        self.X, self.Y = X, Y
        self.i_last = len(X) // 2

    def __call__(self, v, method='hunt'):
        if method == 'hunt':
            i = hunt(self.X, v, self.i_last)
        elif method == 'bisection':
            i = bisection(self.X, v)
        else:
            i = sequential(self.X, v)

        self.i_last = i  # store last index for hunting

        x0, x1 = self.X[i], self.X[i+1]
        y0, y1 = self.Y[i], self.Y[i+1]
        m      = (y1 - y0) / (x1 - x0)
        return y0 + m * (v - x0)

For the rest of this lecture, we use a smooth test function, the
Gaussian
\begin{align}
  f(x) = \exp\left(-\frac{x^2}{2}\right).
\end{align}
<a id="interpolation-error"></a>
To compare interpolation schemes, we measure the interpolation error
\begin{align}
  E(x) \equiv f(x) - P(x),
\end{align}
the difference between the true function $f$ and the interpolant $P$
built from the sampled points.
For polynomial interpolation through $M$ points $x_0, \dots, x_{M-1}$
(linear interpolation is the case $M = 2$), the error takes the form
\begin{align}
  E(x) = \frac{f^{(M)}(\xi)}{M!}\prod_{m=0}^{M-1}(x - x_m)
\end{align}
for some $\xi$ in the function domain
(also known as [interpolation error theorem](https://en.wikipedia.org/wiki/Polynomial_interpolation#Interpolation_error:_Lagrange_remainder_formula)).
The error therefore depends on both the function, through its $M$-th
derivative, and the scheme, through the number and placement of the
points.

Unlike $f_\mathrm{sing}$, the Gaussian function $f$ has no singularities, so
differences in error between interpolation schemes reflect the schemes
themselves rather than the function.

In [ ]:
def f(x):
    return np.exp(-0.5 * x**2)

In [ ]:
import matplotlib.pyplot as plt

Xs = np.sort(np.random.uniform(-5, 5, 20))
Ys = f(Xs)

fi = Interpolator(Xs, Ys)

Xi = np.linspace(min(Xs), max(Xs), 100)
Yi = np.array([fi(x) for x in Xi])

plt.plot(Xs, Ys, 'o-', label='Sampled data')
plt.plot(Xi, Yi, '.',  label='Interpolation')
plt.legend()
plt.show()

Finally, let's test our claim: hunting should be faster than bisection
when queries are sequential and correlated.

In [ ]:
from timeit import timeit

Xs = np.sort(np.random.uniform(-5, 5, 1000))
Ys = f(Xs)
fi = Interpolator(Xs, Ys)
Xi = np.linspace(min(Xs), max(Xs), 10_000)

def job(method):
    Yi = [fi(x, method) for x in Xi]

dt_sequential = timeit("job('sequential')", globals=globals(), number=1)
dt_bisection  = timeit("job('bisection')", globals=globals(), number=1)
dt_hunt       = timeit("job('hunt')",       globals=globals(), number=1)

In [ ]:
print('Sequential:', dt_sequential)
print('Bisection :', dt_bisection)
print('Hunt      :', dt_hunt)

In [ ]:
# HANDSON: change the number of sampled points and interpolation
#          points and measure the performance of all three methods.
#          What are the performance characteristics when
#          N_sample >> N_interpolation and
#          N_sample << N_interpolation?


```{note} Main Take Away

It may seem surprising that interpolation, which sounds like purely
numerical, actually spends much of its effort on searching.
The interpolation formula itself is simple, but locating the right
interval in the data dominates the work.

This may be less surprising if we start asking ourselves what are
computers good at?
Certainly at arithmetic, but equally at bookkeeping (and communication
now when we have internet).
In fact, much of computer science is devoted to studying how to
organize and retrieve data efficiently.
This is what "data structures and algorithms" are about.
Searching in interpolation is a great example of this.
Efficient bookkeeping can matter as much as the numerical formula.

Our codes here were one-dimensional and simple.
But in practice, many scientific applications require multidimensional
interpolation.
In higher dimensions, the search problem quickly becomes far more
complicated.
Efficient methods often rely on tree structures (e.g., kd-trees or
octrees) to organize points and make search practical.
This is the same kind of complexity that appears in $n$-body
simulations and particle-mesh algorithms, where bookkeeping and
efficient search dominate performance.

Thus, linear interpolation serves as a small but important lesson:
computational science (i.e., numerical analysis) and computer science
are deeply connected.
To do science at scale, one must care not only about the equations,
but also about how data are organized and searched.
```

## Polynomial Interpolation and Extrapolation

Given $M$ data points $(x_0, y_0), (x_1, y_1), \dots, (x_{M-1},
y_{M-1})$, there exists a unique polynomial of degree no greater than $M-1$ that
passes through all of them exactly ([Interpolation theorem](https://en.wikipedia.org/wiki/Polynomial_interpolation#Interpolation_theorem)).

### Lagrange's Formula

The classical formula due to Lagrange writes this polynomial as
\begin{align}
  P_{M-1}(x)
  &= \frac{(x-x_1)(x-x_2)\dots(x-x_{M-1})}{(x_0-x_1)(x_0-x_2)\dots(x_0-x_{M-1})} y_0 \\
  &+ \frac{(x-x_0)(x-x_2)\dots(x-x_{M-1})}{(x_1-x_0)(x_1-x_2)\dots(x_1-x_{M-1})} y_1 + \dots \\
  &+ \frac{(x-x_0)(x-x_1)\dots(x-x_{M-2})}{(x_{M-1}-x_0)(x_{M-1}-x_1)\dots(x_{M-1}-x_{M-2})} y_{M-1}.
\end{align}

Using summation and product notation, we can write this more compactly as
\begin{align}
  P_{M-1}(x)
  = \sum_{m=0}^{M-1} y_m \prod_{m'\ne m}\frac{x-x_{m'}}{x_m-x_{m'}}.
\end{align}

It is straightforward to check that the polynomial passes through all
the data points.
Substituting $x = x_k$ for any data point $x_k$, the product equals 1
when $m = k$, and contains the factor $(x_k - x_k) = 0$ when
$m \ne k$.
Hence
\begin{align}
  P_{M-1}(x_k)
  = \sum_{m=0}^{M-1} \delta_{mk} y_m
  = y_k.
\end{align}

### Neville's Algorithm

While Lagrange's formula is mathematically elegant, it is not the most
practical for computation:
* It does not provide an error estimate.
* It is awkward to program, and evaluating it from scratch for every
  degree gives no way to see how the answer improves as more points
  are included.

A better approach is Neville's Algorithm, which builds the
interpolating polynomial recursively, one degree at a time.
Because each degree is obtained from the previous one, the size of the
last improvement naturally provides an error estimate.
Note that Neville's algorithm still costs $\mathcal{O}(M^2)$ operations
per interpolation point, the same as Lagrange's formula; its advantages
are the error estimate and its simple, systematic structure.
This makes it especially useful for small to moderate datasets.

#### Step 1: Zero-Degree Polynomials

A polynomial of degree 0 is just a constant.
For each data point $(x_m, y_m)$, define
\begin{align}
  P_m(x) = y_m
\end{align}
These form the base case for the recursion.

Throughout this section, $f$ is the Gaussian
$f(x) = \exp(-x^2/2)$ defined before the linear interpolation example.

In [ ]:
Xs = np.sort(np.random.uniform(-5, 5, 100))
Ys = f(Xs)

plt.scatter(Xs, Ys, marker='_', color='C0', label=r'$P_m$: polynomials with 0 degree')
plt.xlim(-0.5, 0.5)
plt.ylim( 0.9, 1.05)
plt.legend()

#### Step 2: First-Degree Polynomials

We now combine two neighboring zero-degree polynomials $P_m$ and
$P_{m+1}$ into a first-degree polynomial $P_{m,m+1}(x)$: the line
through $(x_m, y_m)$ and $(x_{m+1}, y_{m+1})$.
It is defined for all $x$: it interpolates for $x_m \le x \le x_{m+1}$
and extrapolates outside this range.
Starting from the two-point form of a line,
\begin{align}
  P_{m,m+1}(x) = \frac{(x - x_{m+1})P_m(x) + (x_m - x)P_{m+1}(x)}{x_m - x_{m+1}}.
\end{align}
It is a
weighted combination of two lower-degree polynomials, with weights
that add up to 1.
It passes through $(x_m, y_m)$ and $(x_{m+1}, y_{m+1})$.

In [ ]:
Pmm1s = []
for m in range(len(Xs)-1):
    Pmm1s.append(lambda x, m=m: ((x - Xs[m+1]) * Ys[m] + (Xs[m] - x) * Ys[m+1]) / (Xs[m] - Xs[m+1]))

In [ ]:
plt.scatter(Xs, Ys, marker='_', color='C0', label=r'$P_m$: polynomials with 0 degree')

label = r'$P_{m,m+1}$: polynomials with 1 degree'
for m, Pmm1 in enumerate(Pmm1s):
    xs = np.linspace(Xs[m], Xs[m+1], 100)
    ys = Pmm1(xs)
    plt.plot(xs, ys, color='C1', label=label)
    label = None

plt.xlim(-0.5, 0.5)
plt.ylim( 0.9, 1.05)
plt.legend()

#### Step 3: Second-Degree Polynomials

Two neighboring first-degree polynomials, $P_{m,m+1}$ and
$P_{m+1,m+2}$, share the middle point $x_{m+1}$.
Combining them in the same way gives the second-degree polynomial
through the three points $x_m$, $x_{m+1}$, and $x_{m+2}$:
\begin{align}
  P_{m,m+1,m+2}
  &= \frac{(x - x_{m+2})P_{m,m+1} + (x_m - x)P_{m+1,m+2}}{x_m - x_{m+2}}.
\end{align}
Like the line in Step 2, it is defined for all $x$.

In [ ]:
Pmm1m2s = []
for m in range(len(Xs)-2):
    Pmm1  = lambda x, m=m: ((x - Xs[m+1]) * Ys[m  ] + (Xs[m  ] - x) * Ys[m+1]) / (Xs[m  ] - Xs[m+1])
    Pm1m2 = lambda x, m=m: ((x - Xs[m+2]) * Ys[m+1] + (Xs[m+1] - x) * Ys[m+2]) / (Xs[m+1] - Xs[m+2])
    Pmm1m2s.append(
        lambda x, m=m, Pmm1=Pmm1, Pm1m2=Pm1m2:
            ((x - Xs[m+2]) * Pmm1(x) + (Xs[m] - x) * Pm1m2(x)) / (Xs[m] - Xs[m+2])
    )

In [ ]:
plt.scatter(Xs, Ys, marker='o', color='C0', label=r'$P_m$: polynomials with 0 degree')

label = r'$P_{m,m+1}$: polynomials with 1 degree'
for m, Pmm1 in enumerate(Pmm1s[:-1]):
    xs = np.linspace(Xs[m], Xs[m+1], 100)
    ys = Pmm1(xs)
    plt.plot(xs, ys, color='C1', label=label)
    label = None

label = r'$P_{m,m+1,m+2}$: polynomials with 2 degree'
for m, Pmm1m2 in enumerate(Pmm1m2s):
    xs = np.linspace(Xs[m], Xs[m+2], 100)
    ys = Pmm1m2(xs)
    plt.plot(xs, ys, ':', color='C2', label=label)
    label = None

plt.xlim(-0.5, 0.5)
plt.ylim( 0.9, 1.05)
plt.legend()

#### Step 4: Third-Degree Polynomials

Repeating the same idea one more time, two neighboring second-degree
polynomials, $P_{m,m+1,m+2}$ and $P_{m+1,m+2,m+3}$, share the two
middle points $x_{m+1}$ and $x_{m+2}$.
Combining them gives the third-degree polynomial through the four
points $x_m, \dots, x_{m+3}$, again defined for all $x$:
\begin{align}
  P_{m,m+1,m+2,m+3}
  &= \frac{(x - x_{m+3})P_{m,m+1,m+2} + (x_m - x)P_{m+1,m+2,m+3}}{x_m - x_{m+3}}.
\end{align}
Note that we no longer need to write out the lower-degree polynomials
by hand: we can build directly on the second-degree polynomials from
Step 3.

In [ ]:
Pmm1m2m3s = []
for m in range(len(Xs)-3):
    Pmm1m2  = Pmm1m2s[m]
    Pm1m2m3 = Pmm1m2s[m+1]
    Pmm1m2m3s.append(
        lambda x, m=m, Pmm1m2=Pmm1m2, Pm1m2m3=Pm1m2m3:
            ((x - Xs[m+3]) * Pmm1m2(x) + (Xs[m] - x) * Pm1m2m3(x)) / (Xs[m] - Xs[m+3])
    )

In [ ]:
plt.scatter(Xs, Ys, marker='o', color='C0', label=r'$P_m$: polynomials with 0 degree')

label = r'$P_{m,m+1}$: polynomials with 1 degree'
for m, Pmm1 in enumerate(Pmm1s[:-2]):
    xs = np.linspace(Xs[m], Xs[m+1], 100)
    ys = Pmm1(xs)
    plt.plot(xs, ys, color='C1', label=label)
    label = None

label = r'$P_{m,m+1,m+2}$: polynomials with 2 degree'
for m, Pmm1m2 in enumerate(Pmm1m2s[:-1]):
    xs = np.linspace(Xs[m], Xs[m+1], 100)
    ys = Pmm1m2(xs)
    plt.plot(xs, ys, ':', color='C2', label=label)
    label = None

label = r'$P_{m,m+1,m+2,m+3}$: polynomials with 3 degree'
for m, Pmm1m2m3 in enumerate(Pmm1m2m3s):
    xs = np.linspace(Xs[m], Xs[m+3], 100)
    ys = Pmm1m2m3(xs)
    plt.plot(xs, ys, '--', color='C3', label=label)
    label = None

plt.xlim(-0.5, 0.5)
plt.ylim( 0.9, 1.05)
plt.legend()

#### Step 5: Recursive Formula

By continuing this process, we arrive at the general recursive formula
for Neville's algorithm (Numerical Recipes Eq. 3.2.3):
\begin{align}
   P_{m,m+1,\dots,m+n}
   &= \frac{(x - x_{m+n})P_{m,m+1,\dots,m+n-1} + (x_{m} - x)P_{m+1,m+2,\dots,m+n} }{x_{m} - x_{m+n}}.
\end{align}
This provides a systematic way to build higher-order interpolating polynomials.

It is helpful to arrange all these polynomials in a tableau, which
Numerical Recipes calls the "family tree".
For $M = 4$ points:
```
x0:  y0 = P0
                 P01
x1:  y1 = P1            P012
                 P12             P0123
x2:  y2 = P2            P123
                 P23
x3:  y3 = P3
```
Each entry is computed from its two "parents" to the left by the
recursive formula above.
All entries are evaluated at the same target $x$, even when $x$ lies
outside the span of an entry's own data points.
For example, with $x$ between $x_1$ and $x_2$, building $P_{0123}(x)$
requires $P_{01}(x)$ and $P_{23}(x)$, both of which are
extrapolations.
The leftmost column holds the data, and the single entry on the right
is the full interpolating polynomial $P_{0,1,\dots,M-1}$.
Moving one column to the right raises the degree by one, either by
stepping *down* (adding a point below) or *up* (adding a point
above).

#### Step 6: Tracking Corrections Instead of Values

Rather than storing the $P$'s themselves, Numerical Recipes
reformulates the recursion in terms of the small differences between
parents and children in the tableau:
\begin{align}
  C_{n,m} &\equiv P_{m,m+1,\dots,m+n} - P_{m,m+1,\dots,m+n-1} \\
  D_{n,m} &\equiv P_{m,m+1,\dots,m+n} - P_{m+1,m+2,\dots,m+n}
\end{align}
In the tableau, $C$ is the correction for a step *down* (the child
gains the point $x_{m+n}$ below) and $D$ is the correction for a step
*up* (the child gains the point $x_m$ above).
Working with these small corrections, rather than subtracting large,
nearly equal $P$'s, keeps round-off errors under control.

Substituting the definitions into the recursive formula gives
\begin{align}
  D_{n+1,m} &= \frac{x_{m+n+1}-x}{x_m - x_{m+n+1}}(C_{n,m+1} - D_{n,m}) \\
  C_{n+1,m} &= \frac{x_m-x}{x_m - x_{m+n+1}}(C_{n,m+1} - D_{n,m})
\end{align}
with $C_{0,m} = D_{0,m} = y_m$.
The $C$'s and $D$'s are exactly the corrections that make the
interpolation one order higher.

#### Step 7: Error Estimate

The final polynomial $P_{0,1,\dots,M-1}$ equals any starting value
$y_m$ plus a set of $C$'s and/or $D$'s that form a path through the
tableau to its rightmost entry.
Each correction along the path is the difference between successive
orders.
Hence the *last* correction added, $\delta y$, measures how much the
answer changed when the last point on the path, either $x_0$ (a final
step up) or $x_{M-1}$ (a final step down), was included.
If the interpolation is converging, the remaining error is of the same
order as $\delta y$, so we use $|\delta y|$ as an estimate of the
[interpolation error](#interpolation-error)
$|E(x)| = |f(x) - P_{0,1,\dots,M-1}(x)|$.
Unlike $E(x)$ itself, which requires knowing the true function $f$,
$\delta y$ is computed from the data alone.

#### Step 8: Implementation

In practice we rarely want a single polynomial through *all* data
points (Runge's phenomenon, see below).
Instead, we choose a polynomial through the $M$ data points closest
to the target, with $M$ small, typically 3–5 points (degree 2–4).
This is where the search routines from the first half of this lecture
come back: we use `hunt` (or `bisection`) to locate the target, then
take a window of $M$ points centered on it.

Within the window, the code follows Numerical Recipes `polint`:
* `ho` and `hp` are $x_m - x$ and $x_{m+n+1} - x$, the numerators in
  the $C$ and $D$ updates; `ho - hp` is the common denominator.
* We start the path at the data point closest to the target, $y_k$.
* At each column, we step down (add $C_{n,k+1}$) or up (add $D_{n,k}$
  and decrement $k$) so that the path stays as close as possible to
  the middle of the tableau.
  Centered paths use points on both sides of the target, which gives
  the most accurate result.
* The last correction is stored in `self.dy` as the error estimate.

In [ ]:
class PolynomialInterpolator:

    def __init__(self, Xs, Ys, M=None):
        if M is None:
            M = len(Xs)

        assert len(Xs) == len(Ys)
        assert 1 <= M <= len(Xs)

        self.Xs, self.Ys, self.M = Xs, Ys, M
        self.i_last = len(Xs) // 2

    def __call__(self, target, search_method='hunt'):
        N, M = len(self.Xs), self.M

        # locate the target and take a window of M points around it
        if search_method == 'hunt':
            i = hunt(self.Xs, target, self.i_last)
        else:
            i = bisection(self.Xs, target)
        self.i_last = i  # store last index for hunting

        j  = max(0, min(N - M, i - (M - 2) // 2))
        Xs = self.Xs[j:j+M]
        Ys = self.Ys[j:j+M]

        C = np.copy(Ys)
        D = np.copy(Ys)

        k = np.argmin(abs(Xs - target))
        y = Ys[k]
        k-= 1

        self.dy = 0
        for n in range(1, M):
            ho  = Xs[:-n] - target
            hp  = Xs[+n:] - target
            w   = C[1:M-n+1] - D[:-n]
            den = ho - hp
            if any(den == 0):
                raise Exception("two input Xs are (to within roundoff) identical.")
            else:
                fac = w / den
            D[:-n] = hp * fac
            C[:-n] = ho * fac

            if 2*(k+1) < (M-n):
                self.dy = C[k+1]
            else:
                self.dy = D[k]
                k -= 1

            y += self.dy

        return y

In [ ]:
Xs = np.linspace(-1,1,21)
Ys = f(Xs)

P  = PolynomialInterpolator(Xs, Ys)

Xi = np.linspace(min(Xs),max(Xs),201)
Yi = []
Ei = []
for x in Xi:
    Yi.append(P(x))
    Ei.append(P.dy)
Yi = np.array(Yi)
Ei = np.array(Ei)

fig, axes = plt.subplots(2,1,figsize=(8,6))
axes[0].scatter(Xs, Ys)
axes[0].plot(Xi, Yi, '-', color='r')
axes[1].semilogy(Xi, abs(Ei))

In [ ]:
# HANDSON*: Change the sampling points to, e.g.,
#
#              Xs = np.linspace(-5,5,21)
#              Xs = -5 * np.cos(np.linspace(0, np.pi, 21))
#
#          How does the error converge if we increase (or decrease)
#          the number of sampling points?
#          What will happen if we increase the size of the domain?
#          (This is called Runge phenomenon.)
#          What will happen if we try to extrapolate?


In [ ]:
# HANDSON*: Our interpolator can take different order of
#          approximations.
#          Create convergence plots and study the error as function of
#          the order.
#          What happen when we have different domain size?
#          What happen when we have different sampling points?

# Same setup as above, but now with a local window of M points
# (polynomial of degree M-1) instead of one polynomial through all points.


## More Interpolation Algorithms

For further study, here are several powerful alternatives and
extensions to polynomial interpolation:
* Cubic Splines:
  Ensure smoothness across intervals by matching derivatives at knots,
  avoiding the oscillations of high-degree polynomials.
* Rational Function Interpolation:
  Use ratios of polynomials to capture asymptotic behavior and improve
  stability.
* Direct Polynomial Coefficients:
  Compute coefficients efficiently (e.g., via Newton's form) when
  polynomials are truly required.
* Multidimensional Interpolation:
  Extend interpolation to higher dimensions, important for
  applications such as spatial data analysis and numerical
  simulations.
* Laplace Interpolation:
  A specialized technique for harmonic functions, using properties of
  Laplace's equation.

Interpolation is not one-size-fits-all.
Polynomial methods are elegant and instructive, but understanding
their strengths and weaknesses is important.
By combining them with more advanced techniques, we can handle a much
wider range of problems in scientific computing.

Instead of re-deriving or implementing all of these algorithms from
scratch, we will try out a few of them using built-in functions from
`numpy` and `scipy`.
This will give us hands-on experience with practical tools while
reinforcing the ideas we have discussed.

### Cubic Splines

The classical choice for smooth interpolation is the cubic spline.
It guarantees continuity of the function, its first derivative, and
its second derivative across intervals.

In [ ]:
from scipy.interpolate import CubicSpline, PchipInterpolator

Xs = np.sort(np.random.uniform(-3, 3, 100))
Ys = f(Xs)

CS    = CubicSpline(Xs, Ys)        # classical cubic spline
Pchip = PchipInterpolator(Xs, Ys)  # monotone, shape-preserving

Xi = np.linspace(-3, 3, 1000)

plt.plot(Xi, f(Xi),     'k',  label='true f(x)')
plt.plot(Xs, Ys,        'o',  label='data')
plt.plot(Xi, CS(Xi),    '--', label='CubicSpline')
plt.plot(Xi, Pchip(Xi), ':',  label='PCHIP')
plt.legend()

plt.xlim(-0.5, 0.5)
plt.ylim( 0.9, 1.05)

In [ ]:
# HANDSON: add some noise to the data and compare `CubicSpline`
#          vs. `PchipInterpolator`.
#          Which one is more robust against oscillations?


### Multidimensional Interpolation

Interpolation often appears in 2D or 3D. SciPy provides two useful
tools:
* `griddata`: works with scattered data.
* `RegularGridInterpolator`: efficient for regular grids.

In [ ]:
from scipy.interpolate import griddata

Xs = np.random.uniform(-3, 3, 100)
Ys = np.random.uniform(-3, 3, 100)
Zs = f(np.sqrt(Xs * Xs + Ys * Ys))

# Interpolate on a grid
Xg, Yg = np.meshgrid(
    np.linspace(-3, 3, 301),
    np.linspace(-3, 3, 301),
    indexing='xy')
Zg = griddata((Xs, Ys), Zs, (Xg, Yg), method='cubic')

plt.imshow(Zg, extent=[-3,3,-3,3], origin='lower')
plt.colorbar(label='Interpolated value')

In [ ]:
# HANDSON: 
#
# 1. Why is the outer shape of the plot irregular?
#
# 2. Adjust the number of sampling points and interpolation points.
#    How does the resulting plots look like?
#
# 3. Try methods 'linear', 'nearest', and 'cubic'.
#    Compare smoothness vs. accuracy.


In [ ]:
# HANDSON: try to use `RegularGridInterpolator` to interpolate on a
#          regular grid.
